# 12 - Testing Integration Patterns

> **Related**: This notebook demonstrates how to use sqlseed in test frameworks (pytest), including fixture patterns, CI/CD integration, and performance benchmarks.

## What You Will Learn

- load_config + fill_from_config test data preparation
- pytest fixture pattern
- preview instead of fill for quick validation
- seed reproducibility testing
- CI/CD integration
- configuration snapshot loading and regeneration
- performance benchmarks

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 01 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| **→ 12** | **Testing Integration Patterns** | **Testing** | **01** |

---

In [ ]:
from __future__ import annotations

# Run from examples/notebooks. Install from the repository root in one resolution:
# python -m pip install -e ".[dev,all]" -e "./plugins/sqlseed-cli" \
#   -e "./plugins/sqlseed-ai[dev,mcp]" -e "./plugins/mcp-server-sqlseed" -e "./plugins/sqlseed-web[dev]"
import os
import sqlite3
import sys
import tempfile
from contextlib import closing
from pathlib import Path

import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

sys.path.insert(0, str(Path("..").resolve()))  # build_demo_db only
from build_demo_db import build

# Keep this object alive across cells. No existing database is opened or rebuilt.
_demo_directory = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-")
demo_root = Path(_demo_directory.name)
os.environ["SQLSEED_CACHE_DIR"] = str(demo_root / "cache")
db_path = build(demo_root / "demo.db")


def require(condition, message):
    """Stop the tutorial if an expected outcome did not occur."""
    if not condition:
        raise RuntimeError(message)


def check_generation(result, count):
    """Check errors and generated row count before showing success."""
    require(not result.errors and result.count == count, f"Generation failed: {result.errors}; count={result.count}")


def read_rows(database, sql):
    """Read actual persisted values using a fixed tutorial query."""
    # The queries below are fixed tutorial SQL, never external identifiers.
    with closing(sqlite3.connect(database)) as connection:
        return connection.execute(sql).fetchall()


with connect(str(db_path), provider="faker") as orch:
    for table, count in (("organizations", 5), ("members", 20), ("projects", 10), ("tags", 8)):
        check_generation(orch.fill_table(table, count=count, seed=42, skip_ai=True), count)

print(f"sqlseed {sqlseed.__version__} | Temporary database: {db_path}")

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Config Loading | `src/sqlseed/config/loader.py` | `load_config()` |

> Corresponding architecture diagram: [§9 Config Model Hierarchy](../../docs/architecture.zh-CN.md#9-配置模型层次结构)

## 1. load_config + fill_from_config Test Data Preparation

Use a config file to batch-prepare test data, ensuring data consistency.

In [ ]:
from sqlseed.config.loader import load_config, save_config
from sqlseed.config.models import GeneratorConfig, TableConfig

# An empty schema allows the parent table to be cleared before dependent rows exist.
config_db = build(demo_root / "config-fixture.db")
test_config = GeneratorConfig(
    db_path=str(config_db),
    provider="faker",
    tables=[
        TableConfig(name="organizations", count=5, clear_before=True, seed=42),
        TableConfig(name="members", count=10, clear_before=True, seed=42),
    ],
)
config_path = demo_root / "test-config.yaml"
save_config(test_config, str(config_path))
loaded = load_config(str(config_path))
require(len(loaded.tables) == 2, "Config did not round-trip")
results = fill_from_config(str(config_path), skip_ai=True)
require(len(results) == 2, "Expected two table results")
for result, count in zip(results, (5, 10), strict=True):
    check_generation(result, count)
require(len(read_rows(config_db, "SELECT org_code FROM organizations")) == 5, "Missing organizations")
require(len(read_rows(config_db, "SELECT member_id FROM members")) == 10, "Missing members")
print("Independent config fixture contains 5 organizations and 10 members.")

## 2. pytest fixture Pattern

Use pytest fixtures to create isolated databases so each test case is independent.

In [ ]:
def create_test_db():
    """A pytest fixture would use tmp_path; here all files live under demo_root."""
    with tempfile.NamedTemporaryFile(suffix=".db", dir=demo_root, delete=False) as temporary:
        path = Path(temporary.name)
    with closing(sqlite3.connect(path)) as connection:
        connection.execute("CREATE TABLE organizations(org_code TEXT PRIMARY KEY, name TEXT NOT NULL)")
        connection.commit()
    return path


def generated_rows(database):
    """Read sorted values for reproducibility comparisons."""
    return read_rows(database, "SELECT org_code, name FROM organizations ORDER BY org_code")


test_db = create_test_db()
test_db2 = create_test_db()
for database in (test_db, test_db2):
    result = fill(str(database), table="organizations", count=3, provider="faker", seed=42, skip_ai=True)
    check_generation(result, 3)
first_rows = generated_rows(test_db)
require(len(first_rows) == 3 and first_rows == generated_rows(test_db2), "Equal seeds did not reproduce values")
print("Two independent fixture databases contain the same three seeded rows:", first_rows)

## 3. preview Instead of fill for Quick Validation

`preview()` does not write to the database, suitable for quickly verifying config correctness.

In [ ]:
before = generated_rows(test_db)
preview_rows = preview(
    str(test_db),
    table="organizations",
    count=3,
    provider="faker",
    seed=42,
    columns={"org_code": {"type": "pattern", "regex": r"ORG-\d{4}"}, "name": {"type": "company"}},
)
require(len(preview_rows) == 3 and generated_rows(test_db) == before, "Preview count/write invariant failed")
print(preview_rows)

## 4. seed Reproducibility Testing

A fixed seed ensures the same data is generated each time, suitable for regression testing.

In [ ]:
seeded_values = []
for _ in range(2):
    result = fill(
        str(test_db), table="organizations", count=3, provider="faker", seed=42, clear_before=True, skip_ai=True
    )
    check_generation(result, 3)
    seeded_values.append(generated_rows(test_db))
require(seeded_values[0] == seeded_values[1] and len(seeded_values[0]) == 3, "Seeded values are not reproducible")
print("Compared actual seeded values after two successful fills:", seeded_values[0])

## 5. CI/CD Integration

Use sqlseed to generate test data in GitHub Actions.

In [ ]:
print("""# Minimal offline project example; create table schemas in pytest fixtures.
name: Test
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v5
      - uses: actions/setup-python@v6
        with:
          python-version: '3.12'
      - run: python -m pip install 'sqlseed==0.2.4' pytest
      - run: pytest tests/
""")
print("For sqlseed repository development, use the simultaneous local package installation in the first cell.")

## 6. Configuration Snapshot Regeneration

`SnapshotManager` saves a configuration and generation parameters, not a copy of the generated data. It supports `save`, `load` and `list_snapshots`; there is no `SnapshotManager.replay`. Load the configuration and regenerate through Core, or use `sqlseed replay` from the CLI package. Reproducibility depends on the schema, provider versions, seed and any time-dependent generators.


In [ ]:
from sqlseed.config.snapshot import SnapshotManager

snap_mgr = SnapshotManager(snapshot_dir=str(demo_root / "snapshots"))
config = GeneratorConfig(
    db_path=str(test_db),
    provider="faker",
    tables=[TableConfig(name="organizations", count=3, clear_before=True, seed=42)],
)
snapshot_path = snap_mgr.save(config, "organizations", 3, seed=42)
snapshot = snap_mgr.load(snapshot_path)
restored = GeneratorConfig.model_validate(snapshot["config"])
require(snapshot["table_name"] == "organizations", "Wrong snapshot table")
replay_config_path = demo_root / "replay-config.yaml"
save_config(restored, str(replay_config_path))
replayed = fill_from_config(str(replay_config_path), skip_ai=True)
require(len(replayed) == 1, "Wrong number of replay results")
check_generation(replayed[0], 3)
require(generated_rows(test_db) == seeded_values[0], "Restored configuration changed seeded values")
print("Snapshot configuration loaded and regenerated three matching rows:", Path(snapshot_path).name)

## 7. Performance Benchmarks

Use MetricsCollector to collect fill performance metrics.

In [ ]:
import time

from sqlseed._utils.metrics import MetricsCollector

metrics = MetricsCollector()
start = time.monotonic()
result = fill(str(test_db), table="organizations", count=100, provider="faker", clear_before=True, skip_ai=True)
elapsed = time.monotonic() - start
check_generation(result, 100)
require(len(generated_rows(test_db)) == 100, "Benchmark did not persist 100 rows")
metrics.record("fill_100_rows", elapsed)
print(f"100 persisted rows in {elapsed:.3f}s; reported throughput {result.rows_per_second:.0f} rows/s")
print(metrics.summary())

## ✅ Summary

| Pattern | Description | Status |
|---|---|---|
| load_config + fill_from_config | Test data preparation | ✅ |
| pytest fixture | Isolated database | ✅ |
| preview instead of fill | Quick validation | ✅ |
| seed reproducibility | Regression testing | ✅ |
| CI/CD integration | GitHub Actions | ✅ |
| Configuration snapshots | Load a saved config and regenerate | ✅ |
| Performance benchmarks | MetricsCollector | ✅ |

**Congratulations!** You have completed all 12 sqlseed tutorials. Review [01-quickstart](01-quickstart.ipynb) or check the [project README](../../README.md) to learn more.

In [ ]:
require(len(generated_rows(test_db)) == 100, "Benchmark row count differs")
require(len(read_rows(config_db, "SELECT member_id FROM members")) == 10, "Config fixture changed")
require(len(read_rows(db_path, "SELECT member_id FROM members")) == 20, "An isolated test changed the demo database")
print("Config, fixtures, seeded values, snapshots, preview and benchmark writes verified.")